<Br><Br>
# Demonstration of the inverse design to suggest formulations of desired Tg with largest biocontent & perform predictions of Tg for a given formulation.
Script written by Rodrigo Q. Albuquerque in April2026 for paper: 

**Natalie Wunder, Rodrigo Q. Albuquerque, Florian Max, and Holger Ruckdäschel, "Accelerating Sustainable Epoxy Resin Development through Bayesian Optimization and Inverse Design", submitted for publication in 2026.**
<Br><Br>

### Import libraries

In [1]:
import pandas as pd
import numpy as np
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, RBF, ConstantKernel, WhiteKernel

### Define functions

In [2]:
def get_formulation_inverse_design(tg_desired = None, bcc_desired = None, tolerance = 1, n_best = 1):
    '''
    This suggests a formulation from the X_virtual dataset that might have a maximum (predicted) Tg
    for any desired biocontent (bcc +/- tolerance), as well as the maximum biocontent for any desired
    Tg (+/- tolerance). The desired value of either tg or bcc must be provided. A total of n_best samples
    are suggested. For that, a gaussian processes model (Matern kernel) is trained using the current 
    features and target.
    Tg is in celsius
    bcc is fractionary (normalized from 0 to 1)
    tolerance is in percentual units

    Return:
    the array with the n_best formulations
    the final dataframe including formulations, bcc and predictd tg

    How to use it (e.g., get the formulation with the highest bcc exhibiting tg of 120):

    >>> formulations, df = get_formulation_inverse_design(tg_desired = 120) 
    '''
    X_virtual = np.load('X_virtual.npy')
    bcc = np.load('bcc_virtual.npy')
    features = np.load('features.npy')
    target = np.load('target.npy')
    labels = ['NC 514 S','LITE 547 LV','PP1402','DER 330','DEN 431','LITE 2402','XFN 1050',
              'IPDA','ECC','Epilox F 1700','XFN 1450','Jeffamine D230','Merginamid L390']

    # train the model
    kernel = ConstantKernel(1.0, constant_value_bounds=(1e-3, 1e3)) * Matern(length_scale=1.0, length_scale_bounds=(1e-3, 1e3), nu=1.5) + WhiteKernel()
    gp = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=10, random_state=42, normalize_y=True)

    # feature scaling not needed, as the normalized features came out of the normalized X_virtual (same units/magnitudes)
    gp.fit(features, target)
    pred = gp.predict(X_virtual)

    # create the pandas DataFrame
    df = pd.DataFrame(X_virtual)
    df.columns = labels
    df['bcc'] = bcc
    df['tg_pred'] = pred

    # apply restrictions
    if tg_desired and not bcc_desired: # highest bcc for a desired Tg
        df1 = df[df['tg_pred'] <= tg_desired * (1 + tolerance/100)]
        df2 = df1[df1['tg_pred'] >= tg_desired * (1 - tolerance/100)]
        dff = df_sorted = df2.sort_values(by='bcc', ascending=False) # order descending
        selection = dff.iloc[:n_best]
    elif bcc_desired and not tg_desired: # highest tg for a desired bcc
        df1 = df[df['bcc'] <= bcc_desired * (1 + tolerance/100)]
        df2 = df1[df1['bcc'] >= bcc_desired * (1 - tolerance/100)]
        dff = df_sorted = df2.sort_values(by='tg_pred', ascending=False) # order descending
        selection = dff.iloc[:n_best]
    elif tg_desired and bcc_desired:
        return 'You must provide EITHER tg_desired OR bcc_desired, but not both. Aborting...'
    else:
        return 'You must provide EITHER tg_desired OR bcc_desired. Aborting...'
        
    formulations = selection.iloc[:, :features.shape[1]].values
    
    return formulations, selection


def perform_prediction(input_array, kernel = 'matern', random_state = 42):
    '''
    This function trains a GP model and performs predictions for the 2D array 'input_array' of shape (n_formulations, 13) 
    Returns:
        predictions and uncertainties
    '''
    X = np.load('features.npy')
    y = np.load('target.npy')
    n_samples = y.shape[0]
    X = X[:n_samples]

    # New sample(s)
    x = np.atleast_2d(input_array)
    
    # preprocessing
    scaler = StandardScaler()
    X_s = scaler.fit_transform(X)
    x_s = scaler.transform(x)

    # kernel
    if kernel == 'matern':
        kernel = ConstantKernel(1.0, constant_value_bounds=(1e-3, 1e3)) * Matern(length_scale=1.0, length_scale_bounds=(1e-3, 1e3), nu=1.5) + WhiteKernel()
    else: # rbf
        kernel = ConstantKernel(1.0, constant_value_bounds=(1e-3, 1e3)) * RBF(length_scale=1.0, length_scale_bounds=(1e-3, 1e3)) + WhiteKernel()

    # model training
    model = GaussianProcessRegressor(kernel = kernel, n_restarts_optimizer=10, random_state=random_state, normalize_y=True)
    model.fit(X_s, y)

    # prediction
    pred, std = model.predict(x_s, return_std = True)

    return pred.reshape(-1), std.reshape(-1)
    

<Br><Br>
# Application 1: inverse design
Example: get 3 formulations with a Tg of 110 Celsius and the highest possible biocontent
<Br><Br>

In [3]:
formulations, full_df = get_formulation_inverse_design(tg_desired = 110, n_best = 3)

In [4]:
full_df

,NC 514 S,LITE 547 LV,PP1402,DER 330,DEN 431,LITE 2402,XFN 1050,IPDA,ECC,Epilox F 1700,XFN 1450,Jeffamine D230,Merginamid L390,bcc,tg_pred
40529,0.0000,0.1758,0.0000,0.0000,0.5945,0.1772,0.0260,0.0204,0.0,0.0000,0.0000,0.0060,0.0000,0.233175,108.960502
28579,0.0011,0.1595,0.0151,0.0070,0.5882,0.2145,0.0018,0.0047,0.0,0.0022,0.0048,0.0000,0.0011,0.229632,109.438825
47884,0.0024,0.1737,0.0012,0.0043,0.5972,0.2204,0.0000,0.0000,0.0,0.0000,0.0000,0.0007,0.0000,0.224360,110.788681


In [5]:
formulations

array([[0.    , 0.1758, 0.    , 0.    , 0.5945, 0.1772, 0.026 , 0.0204,
        0.    , 0.    , 0.    , 0.006 , 0.    ],
       [0.0011, 0.1595, 0.0151, 0.007 , 0.5882, 0.2145, 0.0018, 0.0047,
        0.    , 0.0022, 0.0048, 0.    , 0.0011],
       [0.0024, 0.1737, 0.0012, 0.0043, 0.5972, 0.2204, 0.    , 0.    ,
        0.    , 0.    , 0.    , 0.0007, 0.    ]])

<Br><Br>
# Application 2: inverse design
Example: get 2 formulations with a biocontent of 0.45 (or 45%) and the highest possible Tg
<Br><Br>

In [6]:
formulations, full_df = get_formulation_inverse_design(bcc_desired = 0.45, n_best = 2)

In [7]:
full_df

,NC 514 S,LITE 547 LV,PP1402,DER 330,DEN 431,LITE 2402,XFN 1050,IPDA,ECC,Epilox F 1700,XFN 1450,Jeffamine D230,Merginamid L390,bcc,tg_pred
35284,0.0000,0.3848,0.0192,0.0000,0.3635,0.1595,0.0574,0.0000,0.0000,0.0156,0.0,0.0001,0.0,0.451625,72.327464
46211,0.0002,0.2335,0.0112,0.0005,0.4253,0.1133,0.2118,0.0019,0.0001,0.0023,0.0,0.0000,0.0,0.447503,72.297820


In [8]:
formulations

array([[0.000e+00, 3.848e-01, 1.920e-02, 0.000e+00, 3.635e-01, 1.595e-01,
        5.740e-02, 0.000e+00, 0.000e+00, 1.560e-02, 0.000e+00, 1.000e-04,
        0.000e+00],
       [2.000e-04, 2.335e-01, 1.120e-02, 5.000e-04, 4.253e-01, 1.133e-01,
        2.118e-01, 1.900e-03, 1.000e-04, 2.300e-03, 0.000e+00, 0.000e+00,
        0.000e+00]])

<Br><Br>
# Application 3: simple prediction
Note: The provided formulation should have an stoichiometric ratio (amine:epoxy) in the range 0.8-1.2 (tricky!). We choose therefore the first sample of X_virtual for this example.
<Br><Br>

In [9]:
new_sample = np.load('X_virtual.npy')[0]
new_sample

array([0.000e+00, 7.984e-01, 1.000e-04, 0.000e+00, 0.000e+00, 2.000e-04,
       3.000e-04, 2.000e-04, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00,
       2.008e-01])

In [10]:
predictions, stds = perform_prediction(new_sample)

In [11]:
predictions # in degrees Celsius

array([37.94182118])

In [12]:
stds

array([7.2722309])

In [13]:
# Let's check the biocontent (fractionary, in the range [0,1]) of the first sample of X_virtual
np.load('bcc_virtual.npy')[0] # large bcc (below), small predicted Tg (above)

np.float64(0.8217095999999999)